In [ ]:
# Cell 1: Install dependencies
import subprocess
# Uncomment next line when running on Kaggle:
# subprocess.run(["git", "clone", "https://github.com/YOUR_USER/mini-world-model.git"])
subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"])
print("Setup complete")

In [ ]:
# Cell 2: Collect rollouts
import yaml
from types import SimpleNamespace
from src.data import collect_random_rollouts

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))
collect_random_rollouts(cfg)

In [ ]:
# Cell 3: Train the world model
import yaml
from types import SimpleNamespace
from src.train import train

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))
train(cfg)

In [ ]:
# Cell 4: Plan + generate demo GIF
import torch, yaml
from pathlib import Path
from types import SimpleNamespace
from src.model import build_world_model
from src.env import make_env, get_obs, set_seed
from src.plan import run_mpc_episode
from src.viz import make_demo_gif

with open("config.yaml") as f:
    cfg = SimpleNamespace(**yaml.safe_load(f))

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(cfg.seed)
Path("assets").mkdir(exist_ok=True)

model = build_world_model(cfg).to(device)
model.load_state_dict(torch.load(f"checkpoints/model_ep{cfg.n_epochs:03d}.pt", map_location=device, weights_only=True))
model.eval()

env = make_env(cfg.env_id, cfg.seed)
goal_raw, _ = env.reset(seed=cfg.seed + 99)
goal_obs = torch.tensor(get_obs(goal_raw, cfg.env_id))

frames = run_mpc_episode(model, env, goal_obs, cfg, device)
make_demo_gif(frames, "assets/demo.gif")
print(f"Demo: {len(frames)} frames")

In [ ]:
# Cell 5: Display demo GIF
from IPython.display import Image as IPImage
IPImage(filename="assets/demo.gif")

In [ ]:
# Cell 6: Display loss curves
from IPython.display import Image as IPImage
IPImage(filename="loss_curves.png")